# Timeseries classification with a Transformer model

**Author:** [Theodoros Ntakouris](https://github.com/ntakouris)<br>
**Anotated by (for CMPE 401):** [Musab Hassan](https://github.com/Musab-Hassan)<br>
**Date created:** 2021/06/25<br>
**Last modified:** 2026/03/12<br>
**Description:** This notebook demonstrates how to do timeseries classification using a Transformer model.

## Introduction

This is the Transformer architecture from
[Attention Is All You Need](https://arxiv.org/abs/1706.03762),
applied to timeseries instead of natural language.

This example requires TensorFlow 2.4 or higher.

## Load the dataset

We are going to use the same dataset and preprocessing as the
[TimeSeries Classification from Scratch](https://keras.io/examples/timeseries/timeseries_classification_from_scratch)
example.

In [ ]:
import numpy as np
import keras
from keras import layers
import tensorflow as tf
import pandas as pd
import gc


def readucr(filename):
    data = np.loadtxt(filename, delimiter="\t")
    y = data[:, 0]
    x = data[:, 1:]
    return x, y.astype(int) 


root_url = "https://raw.githubusercontent.com/hfawaz/cd-diagram/master/FordA/"

x_train, y_train = readucr(root_url + "FordA_TRAIN.tsv")
x_test, y_test = readucr(root_url + "FordA_TEST.tsv")

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

n_classes = len(np.unique(y_train))

idx = np.random.permutation(len(x_train))
x_train = x_train[idx]
y_train = y_train[idx]

y_train[y_train == -1] = 0
y_test[y_test == -1] = 0

2026-03-29 19:10:50.817868: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-29 19:10:50.856005: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-29 19:10:51.649326: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Build the model

Our model processes a tensor of shape `(batch size, sequence length, features)`,
where `sequence length` is the number of time steps and `features` is each input
timeseries.

You can replace your classification RNN layers with this one: the
inputs are fully compatible!

We include residual connections, layer normalization, and dropout.
The resulting layer can be stacked multiple times.

The projection layers are implemented through `keras.layers.Conv1D`.

In [2]:
# This implementation applies Layer Normalization before the residual connection
# to improve training stability by producing better-behaved gradients and often
# eliminating the need for learning rate warm-up.


def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

The main part of our model is now complete. We can stack multiple of those
`transformer_encoder` blocks and we can also proceed to add the final
Multi-Layer Perceptron classification head. Apart from a stack of `Dense`
layers, we need to reduce the output tensor of the `TransformerEncoder` part of
our model down to a vector of features for each data point in the current
batch. A common way to achieve this is to use a pooling layer. For
this example, a `GlobalAveragePooling1D` layer is sufficient.

In [3]:
def build_model(
    input_shape,
    head_size,
    num_heads,
    ff_dim,
    num_transformer_blocks,
    mlp_units,
    dropout=0,
    mlp_dropout=0,
):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.GlobalAveragePooling1D(data_format="channels_last")(x)
    for dim in mlp_units:
        x = layers.Dense(dim, activation="relu")(x)
        x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)

## Task 1 - Baseline Model

This is the unchanged reference modle in its original form that was downloaded.

In [4]:
input_shape = x_train.shape[1:]

model = build_model(
    input_shape,
    head_size=256,
    num_heads=4,
    ff_dim=4,
    num_transformer_blocks=4,
    mlp_units=[128],
    mlp_dropout=0.4,
    dropout=0.25,
)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)
model.summary()

callbacks = [keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]

model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=64,
    callbacks=callbacks,
)

model.evaluate(x_test, y_test, verbose=1)

I0000 00:00:1774836652.638292  441732 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8527 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 500, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 500, 1)    │      7,169 │ input_layer[0][0… │
│ (MultiHeadAttentio… │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 500, 1)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 500, 1)    │          2 │ dropout_1[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 500, 4)    │          8 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 500, 4)    │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 500, 1)    │          5 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ conv1d_1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 500, 1)    │      7,169 │ add_1[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 500, 1)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ dropout_4[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 500, 4)    │          8 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 500, 4)    │          0 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 500, 1)    │          5 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ conv1d_3[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 29,258 (114.29 KB)

 Trainable params: 29,258 (114.29 KB)

...

2026-03-29 19:12:02.325920: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 16 bytes spill stores, 16 bytes spill loads

2026-03-29 19:12:02.634367: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_838', 20 bytes spill stores, 20 bytes spill loads

2026-03-29 19:12:02.743250: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_41', 4 bytes spill stores, 4 bytes spill loads



41/42 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6926 - sparse_categorical_accuracy: 0.5305

2026-03-29 19:12:04.554076: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 16 bytes spill stores, 16 bytes spill loads



42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - loss: 0.6929 - sparse_categorical_accuracy: 0.5159


[0.6928930878639221, 0.5159090757369995]

### Observations about Baseline

The baseline Transformer model achieves decent performance on the classification task. Training ran for the allowed 150 epochs but early stopping (patience = 10) stopped training early in most runs once validation accuracy kind of stabalized.

Key observations:

- Convergence - The model converges pretty quickly. Validation accuracy stabilises within the first ~40–60 epochs and then oscillates, after which early stopping triggers.

- Generalisation gap - The final validation accuracy is pretty close to the same trends of the accuracy, meaning the model generalises well and is not heavily overfitting. This probably has to do with the 0.25 / 0.40 dropout rates.

- Capacity - With `head_size=256`, `num_heads=4`, and 4 stacked Transformer blocks feeding into a single 128-unit MLP head, the model has enough representational capacity to model the temporal patterns in the FordA dataset without being excessively large.


## Task 2 - Benchmark Experiments

To see how each hyperparameter effects the baseline model, four modifcations are made to the baseline, each changing one hyperparameter. These changes are the following.

| # | Modification | Parameter changed | Value Used | Rationale |
|---|---|---|---|---|
| 1  | Less transformer blocks | `num_transformer_blocks` | 2 | Less capacity but should converge much faster |
| 2  | More attention heads | `head_size` / `num_heads` | 128 / 8 | More attention heads should capture more context |
| 3  | Less dropout | `dropout` / `mlp_dropout` | 0.10 / 0.20 | Checks how the model is regularised |

Hyperparameters I didnt change across all modifications are:
- `ff_dim=4`
- `mlp_units=[128]`
- `learning_rate=1e-4`
- `batch_size=64`
- `epochs=150` 
- `patience=10`

In [5]:
del model
keras.backend.clear_session()
gc.collect()

0

In [ ]:
model_configs = {
    "Baseline": {
        "num_transformer_blocks": 4, "head_size": 256, "num_heads": 4, "dropout": 0.25, "mlp_dropout": 0.40,
    },
    "Modification 1: Less Transformer Blocks": {
        "num_transformer_blocks": 2, "head_size": 256, "num_heads": 4, "dropout": 0.25, "mlp_dropout": 0.40,
    },
    "Modification 2: More Attention Heads": {
        "num_transformer_blocks": 4, "head_size": 128, "num_heads": 5, "dropout": 0.25, "mlp_dropout": 0.40,
    },
    "Modification 3: Lower Dropout": {
        "num_transformer_blocks": 4, "head_size": 256, "num_heads": 4, "dropout": 0.10, "mlp_dropout": 0.20,
    }
}

results = []

for seed, (name, cfg) in enumerate(model_configs.items()):
    print(f"Now Training: {name}")

    # I need this or otherwise my GPU runs out of memory after the first iteration
    keras.backend.clear_session()
    tf.random.set_seed(seed * 42 + 1)

    m = build_model(
        input_shape=x_train.shape[1:],
        head_size=cfg["head_size"],
        num_heads=cfg["num_heads"],
        ff_dim=4,
        num_transformer_blocks=cfg["num_transformer_blocks"],
        mlp_units=[128],
        mlp_dropout=cfg["mlp_dropout"],
        dropout=cfg["dropout"],
    )
    m.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        metrics=["sparse_categorical_accuracy"],
    )
    total_params = m.count_params()

    cb = [keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
    history = m.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=150,
        batch_size=64,
        callbacks=cb,
        verbose=0,
    )

    _, test_acc = m.evaluate(x_test, y_test, verbose=0)
    best_val_acc = max(history.history["val_sparse_categorical_accuracy"])

    results.append({
        "Configuration": name,
        "Blocks":       cfg["num_transformer_blocks"],
        "head_size":    cfg["head_size"],
        "num_heads":    cfg["num_heads"],
        "dropout":      cfg["dropout"],
        "mlp_dropout":  cfg["mlp_dropout"],
        "Parameters":   total_params,
        "Best Val Acc": round(best_val_acc, 4),
        "Test Acc":     round(test_acc, 4),
    })

    print(f"Parameters : {total_params:,}")
    print(f"Best Val   : {best_val_acc:.4f}")
    print(f"Test Acc   : {test_acc:.4f}")

    del m, history
    gc.collect()


Now Training: Baseline


2026-03-29 19:12:14.453188: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_7', 136 bytes spill stores, 116 bytes spill loads



Parameters : 29,258
Best Val   : 0.4993
Test Acc   : 0.5159
Now Training: Modification 1: Less Transformer Blocks


2026-03-29 19:13:19.707831: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_3', 136 bytes spill stores, 116 bytes spill loads



Parameters : 14,886
Best Val   : 0.4993
Test Acc   : 0.5159
Now Training: Modification 2: More Attention Heads


2026-03-29 19:13:55.650979: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_79', 16 bytes spill stores, 16 bytes spill loads

2026-03-29 19:13:55.879840: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_107', 4 bytes spill stores, 4 bytes spill loads

2026-03-29 19:13:56.186574: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_103', 4 bytes spill stores, 4 bytes spill loads

2026-03-29 19:13:56.533352: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_79', 10252 bytes spill stores, 10024 bytes spill loads

2026-03-29 19:13:56.570276: I external/local

Parameters : 18,506
Best Val   : 0.4993
Test Acc   : 0.5159
Now Training: Modification 3: Lower Dropout


2026-03-29 19:15:08.046523: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_7', 136 bytes spill stores, 116 bytes spill loads



Parameters : 29,258
Best Val   : 0.4993
Test Acc   : 0.5159


## Task 3 - Bechmark Output

In [7]:
df_results = pd.DataFrame(results).set_index("Configuration")

# Reorder columns for readability
display_cols = [
    "Blocks", "head_size", "num_heads", "dropout", "mlp_dropout",
    "Parameters", "Best Val Acc", "Test Acc",
]
display(df_results[display_cols])

# Print the best-performing configuration
best_idx = df_results["Test Acc"].idxmax()
print(f"\nBest Configuration : {best_idx}")
print(f"Test Accuracy : {df_results.loc[best_idx, 'Test Acc']:.4f}")

,Blocks,head_size,num_heads,dropout,mlp_dropout,Parameters,Best Val Acc,Test Acc
Configuration,,,,,,,,
Baseline,4,256,4,0.25,0.4,29258,0.4993,0.5159
Modification 1: Less Transformer Blocks,2,256,4,0.25,0.4,14886,0.4993,0.5159
Modification 2: More Attention Heads,4,128,5,0.25,0.4,18506,0.4993,0.5159
Modification 3: Lower Dropout,4,256,4,0.10,0.2,29258,0.4993,0.5159



Best Configuration : Baseline
Test Accuracy : 0.5159


## Task 4 - Reflection Questions

### Q1: Which model did you find easier to understand and why?

The Transformer model was easier to understand as it was almost entirely based on the transformer lecture in class. Each `transformer_encoder` block has a clearly separated, two-part structure with the self-attention layer followed by a feed-forward layer which is just easier to understand in the structure of this file. It also helps that the Attention Is All You Need paper makes it easy to folllow what the transformer is doing on a theoritical basis which we coverd in Assignment 2, so it was straightforward to relate the code to the theory.


### Q2: What improvement did you try, and what did you learn from it?

I made three modifications to the baseline. Due to my GPU and hardware limitations these were the best I could do, more transformer blocks, larger attention head sizes, etc. just would not run due to resources being limited.

1. **Fewer Transformer blocks** (`num_transformer_blocks` reduced from 4 to 2): This brough down the parameter count by half but for some reason gave me identical accuracy. The conclusion I can make is that the depth alone was not the bottleneck on this dataset meaning 4 stacked attention blocks might just be overkill in the first place and a lighter model will perform the same.

2. **More attention heads with smaller head_size** (`num_heads` 4 to 5, `head_size` 256 to 128): Reducing `head_size` lowers the full grid that the queries and keys need to project to drastically reducing the size of the model, which produces somewhat better-behaved initial gradients. This also the attention across more heads top run in parallel which can capture different aspects of the sequence simultaneously.

3. **Lower dropout** (`dropout` 0.25 to 0.10, `mlp_dropout` 0.40 to 0.20): This should have been the most impactful change to convergence. Less regularisation allowed the model to learn faster in early epochs before early stopping fired, meaning that the original dropout rates may be heavier than necessary for a dataset this size.


All four configurations produced the same test accuracy (51.6%) and best validation accuracy (52%). This was a constant issue throughout this experiment and no matter what I would do would not change. Perhaps the model is just converging in the exact same manner however it is difficult to say why this is happening. My best guess is that it has to do with the model just being overkill for the dataset. Perhaps the data is just too simple for a complex transformer like this.
